# Multi-Class traffic assignment demo

In this example, we grab a pre-existing model and do the following:

* Add new fields to the network
* Re-compute fields in the network
* Visualize the network
* Perform Skimming
* Perform Assignment


## Running on Google Colab

Press here to open this notebook in Google Colab <a href="https://colab.research.google.com/github/outerl/AequilibraE-demo/blob/main/multi_class_traffic_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Once in Colab, uncomment the cell below and run it.

**It will restart your Python environment to load all packages correctly**, so it is not possible just run all cells

In [ ]:
# !apt-get update && apt-get install libsqlite3-mod-spatialite
# !apt-get install -y libspatialite-dev
# !pip install numpy --upgrade
# !pip install aequilibrae matplotlib
# exit()

# Before we begin
Let's make a copy of the model so we can make the changes we want without overwriting the original model

If you have are running this notebook on Google Colab and want to save your model into your Google Drive, you should:

1. Un-comment and run the two cells below
2. Accept the terms and conditions when asking to connect Colab to Google Drive.
3. Skip the following cell, where we copy the model around

In [ ]:
# #Just in case you wabt to use Google colab
# from google.colab import drive
# drive.mount("/content/gdrive")

In [ ]:
# !wget https://github.com/outerl/AequilibraE-demo/releases/download/Freeworld/LongAn_base_model.zip
# import zipfile
# with zipfile.ZipFile('LongAn_base_model.zip',"r") as zip_ref:
#     zip_ref.extractall(".")

# # You probably want to move this into your google drive, but we will leave in iun the temporary folder of the VM running the model
# model_path = "./LongAn_base_model"

If not using Colab, you will need to download the [MODEL](https://github.com/outerl/AequilibraE-demo/releases/download/Freeworld/LongAn_base_model.zip), unzip it and set the folders below accordingly

## Let's do some modeling

In [ ]:
# Imports
from pathlib import Path
from aequilibrae import Project

In [ ]:
# We open a project
project = Project.from_path(Path(model_path))

## Model stats

In [ ]:
print(f"Links: {project.network.count_links():,}")
print(f"Nodes: {project.network.count_nodes():,}")
print(f"Zones: {project.network.count_centroids():,}")

Let's see which fields we have in our links layer

In [ ]:
project.network.links.fields.all_fields()

We are missing travel time. Let's add it

In [ ]:
fields = project.network.links.fields
if "ff_ttime_ab" not in project.network.links.fields.all_fields():
    fields.add("ff_ttime_ab", description="AB Free flow travel time")
    fields.add("ff_ttime_ba", description="BA Free flow travel time")
    fields.save()

In [ ]:
project.network.links.fields.all_fields()

#### Let's use some SQL to compute these fields

Speeds are in KM/h and distances are ALWAYS in meters in AequilibraE


In [ ]:
%%time
sql = """Update links set ff_ttime_ab=(distance/1000)/speed_ab * 60, 
                          ff_ttime_ba=(distance/1000)/speed_ab * 60"""
with project.db_connection_spatial as conn:
    conn.execute(sql)

In [ ]:
links = project.network.links.data
nodes = project.network.nodes.data
zones = project.zoning.data

In [ ]:
links.head(3)

In [ ]:
nodes.head(3)

In [ ]:
zones.tail(3)

## Computes Graph



In [ ]:
%%time
project.network.build_graphs(modes=["c"])

In [ ]:
graph = project.network.graphs["c"]
graph.set_graph("ff_ttime")
graph.set_skimming(["distance", "ff_ttime"])

## Matrices

In [ ]:
project.matrices.list()

In [ ]:
mat = project.matrices.get_matrix("base_demand")
mat.names

In [ ]:
project.matrices.update_database()
project.matrices.list()

## Assignment

Let's assign only a few of the matrices we exported

In [ ]:
demand_cars = project.matrices.get_matrix("base_demand")
demand_cars.computational_view(['red_cars_AM', 'blue_cars_AM'])

In [ ]:
demand_trucks = project.matrices.get_matrix("base_demand")
demand_trucks.computational_view(['Trucks_AM'])

In [ ]:
from aequilibrae.paths import TrafficAssignment, TrafficClass

In [ ]:

# Create the assignment class
car_class = TrafficClass(name="car", graph=graph, matrix=demand_cars)
truck_class = TrafficClass(name="truck", graph=graph, matrix=demand_trucks)

truck_class.set_pce(1.8)
# truck_class.set_select_links()
# truck_class.set_vot()
# truck_class.set_fixed_cost()

In [ ]:

assig = TrafficAssignment()

# The first thing to do is to add at list of traffic classes to be assigned
assig.add_class(car_class)
assig.add_class(truck_class)

# We set these parameters only after adding one class to the assignment
assig.set_vdf("BPR")  # This is not case-sensitive

# Then we set the volume delay function
assig.set_vdf_parameters({"alpha": 0.15, "beta": 4.0})  # And its parameters

assig.set_capacity_field("capacity")  # The capacity and free flow travel times as they exist in the graph
assig.set_time_field("ff_ttime")

# And the algorithm we want to use to assign
assig.set_algorithm("bfw")

# Since I haven't checked the parameters file, let's make sure convergence criteria is good
assig.max_iter = 5
assig.rgap_target = 0.0001

assig.execute()  # we then execute the assignment

In [ ]:
assig.report()

In [ ]:
assig.results().max()

In [ ]:
assig.save_results("tutorial")